In [1]:
!pip install ultralytics supervision fastapi uvicorn nest-asyncio pyngrok -q
print("Ready")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
Ready


In [2]:
from google.colab import files
print("Upload Store 1 files: CAM 1 - zone.mp4, CAM 2 - zone.mp4, CAM 3 - entry.mp4, CAM 5 - billing.mp4, Store 1 - layout.png")
uploaded = files.upload()


Upload Store 1 files: CAM 1 - zone.mp4, CAM 2 - zone.mp4, CAM 3 - entry.mp4, CAM 5 - billing.mp4, Store 1 - layout.png


Saving CAM 1 - zone.mp4 to CAM 1 - zone.mp4
Saving CAM 2 - zone.mp4 to CAM 2 - zone.mp4
Saving CAM 3 - entry.mp4 to CAM 3 - entry.mp4
Saving CAM 5 - billing.mp4 to CAM 5 - billing.mp4
Saving Store 1 - layout.png to Store 1 - layout.png


In [4]:
print("Upload Store 2 files: billing_area.mp4, zone.mp4, entry 1.mp4, entry 2.mp4, store 2 - layout.png")
uploaded2 = files.upload()

Upload Store 2 files: billing_area.mp4, zone.mp4, entry 1.mp4, entry 2.mp4, store 2 - layout.png


Saving billing_area.mp4 to billing_area.mp4
Saving zone.mp4 to zone.mp4
Saving entry 1.mp4 to entry 1.mp4
Saving entry 2.mp4 to entry 2.mp4
Saving store 2 - layout.png to store 2 - layout.png


In [5]:
print("Upload: POS sample transactions CSV + sample_events.jsonl")
uploaded3 = files.upload()

Upload: POS sample transactions CSV + sample_events.jsonl


Saving sample_eventsbe42122.jsonl to sample_eventsbe42122.jsonl
Saving POS - sample transactionsb1e826f.csv to POS - sample transactionsb1e826f.csv


In [6]:
import os
all_files = os.listdir(".")
print("Files in Colab:")
for f in sorted(all_files):
    if not f.startswith(".") and not f.endswith(".pt"):
        size = os.path.getsize(f) // 1024
        print(f"  {f}  ({size} KB)")

Files in Colab:
  CAM 1 - zone.mp4  (176048 KB)
  CAM 2 - zone.mp4  (158420 KB)
  CAM 3 - entry.mp4  (186372 KB)
  CAM 5 - billing.mp4  (71549 KB)
  POS - sample transactionsb1e826f.csv  (5 KB)
  Store 1 - layout.png  (366 KB)
  billing_area.mp4  (48231 KB)
  entry 1.mp4  (26747 KB)
  entry 2.mp4  (40388 KB)
  sample_data  (4 KB)
  sample_eventsbe42122.jsonl  (4 KB)
  store 2 - layout.png  (193 KB)
  zone.mp4  (49876 KB)


In [7]:
import numpy as np

# Based on store layouts and camera positions we identified:
# Store 1: rectangular store, entry on left, cash counter on right
# Store 2: wider store, entrance at bottom center

STORE_CONFIG = {
    "ST1": {
        "store_id": "ST1008",
        "cameras": {
            "CAM1": {
                "file": "CAM 1 - zone.mp4",
                "type": "zone",
                "zones": {
                    "left_wall_brands": {
                        "polygon": np.array([[0,0],[960,0],[960,540],[0,540]]),
                        "zone_id": "ST1008_Z01", "zone_type": "SHELF",
                        "zone_name": "Left Wall Brands", "is_revenue_zone": "Yes"
                    },
                    "foh_center": {
                        "polygon": np.array([[0,540],[960,540],[960,1080],[0,1080]]),
                        "zone_id": "ST1008_Z02", "zone_type": "DISPLAY",
                        "zone_name": "FOH Center", "is_revenue_zone": "Yes"
                    },
                }
            },
            "CAM2": {
                "file": "CAM 2 - zone.mp4",
                "type": "zone",
                "zones": {
                    "right_wall_brands": {
                        "polygon": np.array([[0,0],[960,0],[960,540],[0,540]]),
                        "zone_id": "ST1008_Z03", "zone_type": "SHELF",
                        "zone_name": "Right Wall Brands", "is_revenue_zone": "Yes"
                    },
                    "makeup_unit": {
                        "polygon": np.array([[0,540],[960,540],[960,1080],[0,1080]]),
                        "zone_id": "ST1008_Z04", "zone_type": "DISPLAY",
                        "zone_name": "Makeup Unit", "is_revenue_zone": "Yes"
                    },
                }
            },
            "CAM3": {
                "file": "CAM 3 - entry.mp4",
                "type": "entry",
                "zones": {
                    "entry_gate": {
                        "polygon": np.array([[600,600],[1300,600],[1300,1080],[600,1080]]),
                        "zone_id": "ST1008_Z_ENTRY", "zone_type": "ENTRY",
                        "zone_name": "Store Entry", "is_revenue_zone": "No"
                    },
                }
            },
            "CAM5": {
                "file": "CAM 5 - billing.mp4",
                "type": "billing",
                "zones": {
                    "billing_queue": {
                        "polygon": np.array([[100,50],[860,50],[860,700],[100,700]]),
                        "zone_id": "ST1008_Z_BILLING", "zone_type": "BILLING",
                        "zone_name": "Billing Counter Queue", "is_revenue_zone": "Yes"
                    },
                }
            },
        }
    },
    "ST2": {
        "store_id": "ST1009",
        "cameras": {
            "CAM1": {
                "file": "zone.mp4",
                "type": "zone",
                "zones": {
                    "main_zone": {
                        "polygon": np.array([[0,0],[1920,0],[1920,540],[0,540]]),
                        "zone_id": "ST1009_Z01", "zone_type": "SHELF",
                        "zone_name": "Main Zone", "is_revenue_zone": "Yes"
                    },
                    "gondola_area": {
                        "polygon": np.array([[0,540],[1920,540],[1920,1080],[0,1080]]),
                        "zone_id": "ST1009_Z02", "zone_type": "DISPLAY",
                        "zone_name": "Gondola Area", "is_revenue_zone": "Yes"
                    },
                }
            },
            "CAM_E1": {
                "file": "entry 1.mp4",
                "type": "entry",
                "zones": {
                    "entry_left": {
                        "polygon": np.array([[0,400],[960,400],[960,1080],[0,1080]]),
                        "zone_id": "ST1009_Z_ENTRY1", "zone_type": "ENTRY",
                        "zone_name": "Entry Left", "is_revenue_zone": "No"
                    },
                }
            },
            "CAM_E2": {
                "file": "entry 2.mp4",
                "type": "entry",
                "zones": {
                    "entry_right": {
                        "polygon": np.array([[0,400],[960,400],[960,1080],[0,1080]]),
                        "zone_id": "ST1009_Z_ENTRY2", "zone_type": "ENTRY",
                        "zone_name": "Entry Right", "is_revenue_zone": "No"
                    },
                }
            },
            "CAM_B": {
                "file": "billing_area.mp4",
                "type": "billing",
                "zones": {
                    "billing_queue": {
                        "polygon": np.array([[100,50],[860,50],[860,900],[100,900]]),
                        "zone_id": "ST1009_Z_BILLING", "zone_type": "BILLING",
                        "zone_name": "Billing Counter Queue", "is_revenue_zone": "Yes"
                    },
                }
            },
        }
    }
}

print(" Zone config ready")
for store, config in STORE_CONFIG.items():
    print(f"\n{store} ({config['store_id']}):")
    for cam, c in config["cameras"].items():
        print(f"  {cam}: {c['file']} → {list(c['zones'].keys())}")

 Zone config ready

ST1 (ST1008):
  CAM1: CAM 1 - zone.mp4 → ['left_wall_brands', 'foh_center']
  CAM2: CAM 2 - zone.mp4 → ['right_wall_brands', 'makeup_unit']
  CAM3: CAM 3 - entry.mp4 → ['entry_gate']
  CAM5: CAM 5 - billing.mp4 → ['billing_queue']

ST2 (ST1009):
  CAM1: zone.mp4 → ['main_zone', 'gondola_area']
  CAM_E1: entry 1.mp4 → ['entry_left']
  CAM_E2: entry 2.mp4 → ['entry_right']
  CAM_B: billing_area.mp4 → ['billing_queue']


In [8]:
import cv2
import json
import uuid
import numpy as np
from ultralytics import YOLO
import supervision as sv
from collections import defaultdict
from datetime import datetime, timedelta
import random

model = YOLO("yolov8n.pt")

SAMPLE_EVERY = 3

def mock_demographics():
    genders = ["F","F","F","M","F","M"]
    buckets = [("18-24",21),("25-34",28),("25-34",31),("35-44",38),("18-24",22)]
    g = random.choice(genders)
    bucket, age = random.choice(buckets)
    return g, age, bucket

def process_camera(store_id, cam_id, cam_config, base_ts):
    filename = cam_config["file"]
    cam_type = cam_config["type"]
    zones    = cam_config["zones"]

    if not os.path.exists(filename):
        print(f"    {filename} not found — skipping")
        return [], {}

    print(f"   {cam_id} ({filename})...", end=" ")

    zone_objects = {
        name: sv.PolygonZone(polygon=z["polygon"])
        for name, z in zones.items()
    }

    tracker   = sv.ByteTrack()
    box_ann   = sv.BoxAnnotator()
    label_ann = sv.LabelAnnotator()

    track_first_seen = {}
    track_demographics = {}
    track_zone_entry = {}       # (tid, zone_name) -> entry_ts
    zone_dwell  = defaultdict(lambda: defaultdict(int))
    zone_counts = defaultdict(list)

    # Queue specific state
    queue_join_info = {}  # tid -> {join_ts, position}
    queue_position  = 0
    prev_billing_ids = set()

    cap   = cv2.VideoCapture(filename)
    fps   = cap.get(cv2.CAP_PROP_FPS) or 25
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    out = cv2.VideoWriter(
        f"output_{store_id}_{cam_id}.mp4",
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps, (w, h)
    )

    events     = []
    frame_idx  = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1
        ts = base_ts + timedelta(seconds=frame_idx / fps)
        ts_str = ts.isoformat()

        if frame_idx % SAMPLE_EVERY != 0:
            out.write(frame)
            continue

        results    = model(frame, classes=[0], verbose=False)[0]
        detections = sv.Detections.from_ultralytics(results)
        detections = tracker.update_with_detections(detections)

        current_ids = set()
        if detections.tracker_id is not None:
            current_ids = set(detections.tracker_id.tolist())

        # ── New person detected → entry event ──────────────────
        for tid in current_ids:
            if tid not in track_first_seen:
                track_first_seen[tid] = ts_str
                g, age, bucket = mock_demographics()
                track_demographics[tid] = (g, age, bucket)

                if cam_type == "entry":
                    events.append({
                        "event_type":      "entry",
                        "id_token":        f"ID_{60000 + tid}",
                        "store_code":      store_id,
                        "camera_id":       cam_id,
                        "event_timestamp": ts_str,
                        "is_staff":        False,
                        "gender_pred":     g,
                        "age_pred":        age,
                        "age_bucket":      bucket,
                        "is_face_hidden":  False,
                        "group_id":        None,
                        "group_size":      None,
                    })

        # ── Person left frame → exit event ─────────────────────
        gone_ids = set(track_first_seen.keys()) - current_ids
        for tid in list(gone_ids):
            if cam_type == "entry" and tid in track_demographics:
                g, age, bucket = track_demographics[tid]
                events.append({
                    "event_type":      "exit",
                    "id_token":        f"ID_{60000 + tid}",
                    "store_code":      store_id,
                    "camera_id":       cam_id,
                    "event_timestamp": ts_str,
                    "is_staff":        False,
                    "gender_pred":     g,
                    "age_pred":        age,
                    "age_bucket":      bucket,
                    "is_face_hidden":  False,
                    "group_id":        None,
                    "group_size":      None,
                })
            del track_first_seen[tid]
            track_demographics.pop(tid, None)

        # ── Zone events ─────────────────────────────────────────
        current_billing_ids = set()

        for zone_name, zone_obj in zone_objects.items():
            zone_cfg  = zones[zone_name]
            mask      = zone_obj.trigger(detections=detections)
            ids_in_zone = set()

            if detections.tracker_id is not None:
                ids_in_zone = set(detections.tracker_id[mask].tolist())

            # Zone entered
            for tid in ids_in_zone:
                key = (tid, zone_name)
                if key not in track_zone_entry:
                    track_zone_entry[key] = ts
                    g, age, bucket = track_demographics.get(tid, mock_demographics())

                    if zone_cfg["zone_type"] == "BILLING":
                        queue_position += 1
                        queue_join_info[tid] = {
                            "join_ts":   ts_str,
                            "position":  queue_position,
                            "zone_cfg":  zone_cfg,
                            "cam_id":    cam_id,
                            "gender":    g,
                            "age":       age,
                            "bucket":    bucket,
                        }
                        current_billing_ids.add(tid)
                    else:
                        # Get centroid of detection
                        if detections.tracker_id is not None:
                            idx_list = detections.tracker_id.tolist()
                            if tid in idx_list:
                                idx = idx_list.index(tid)
                                box = detections.xyxy[idx]
                                cx  = float((box[0]+box[2])/2)
                                cy  = float((box[1]+box[3])/2)
                            else:
                                cx, cy = 0.0, 0.0
                        else:
                            cx, cy = 0.0, 0.0

                        events.append({
                            "event_type":      "zone_entered",
                            "track_id":        tid,
                            "store_id":        store_id,
                            "camera_id":       cam_id,
                            "zone_id":         zone_cfg["zone_id"],
                            "zone_name":       zone_cfg["zone_name"],
                            "zone_type":       zone_cfg["zone_type"],
                            "is_revenue_zone": zone_cfg["is_revenue_zone"],
                            "event_time":      ts_str,
                            "zone_hotspot_x":  round(cx, 1),
                            "zone_hotspot_y":  round(cy, 1),
                            "gender":          g,
                            "age":             age,
                            "age_bucket":      bucket,
                        })

                zone_dwell[zone_name][tid] += SAMPLE_EVERY

            # Zone exited
            exited = {
                k for k in track_zone_entry
                if k[1] == zone_name and k[0] not in ids_in_zone
            }
            for key in exited:
                tid = key[0]
                entry_ts = track_zone_entry.pop(key)
                dwell = (ts - entry_ts).total_seconds()
                g, age, bucket = track_demographics.get(tid, mock_demographics())

                if zone_cfg["zone_type"] == "BILLING" and tid in queue_join_info:
                    info = queue_join_info.pop(tid)
                    served_ts = (entry_ts + timedelta(seconds=min(dwell*0.1, 15))).isoformat()
                    abandoned = dwell > 120
                    events.append({
                        "queue_event_id":         str(uuid.uuid4()),
                        "event_type":             "queue_abandoned" if abandoned else "queue_completed",
                        "track_id":               tid,
                        "store_id":               store_id,
                        "camera_id":              cam_id,
                        "zone_id":                zone_cfg["zone_id"],
                        "zone_name":              zone_cfg["zone_name"],
                        "zone_type":              "BILLING",
                        "is_revenue_zone":        "Yes",
                        "queue_join_ts":          info["join_ts"],
                        "queue_served_ts":        None if abandoned else served_ts,
                        "queue_exit_ts":          ts_str,
                        "wait_seconds":           round(dwell),
                        "queue_position_at_join": info["position"],
                        "abandoned":              abandoned,
                        "zone_hotspot_x":         0.0,
                        "zone_hotspot_y":         0.0,
                        "gender":                 g,
                        "age":                    age,
                        "age_bucket":             bucket,
                    })
                else:
                    events.append({
                        "event_type":      "zone_exited",
                        "track_id":        tid,
                        "store_id":        store_id,
                        "camera_id":       cam_id,
                        "zone_id":         zone_cfg["zone_id"],
                        "zone_name":       zone_cfg["zone_name"],
                        "zone_type":       zone_cfg["zone_type"],
                        "is_revenue_zone": zone_cfg["is_revenue_zone"],
                        "event_time":      ts_str,
                        "zone_hotspot_x":  0.0,
                        "zone_hotspot_y":  0.0,
                        "gender":          g,
                        "age":             age,
                        "age_bucket":      bucket,
                    })

            # Track zone counts
            zone_counts[zone_name].append({
                "timestamp": round(frame_idx/fps, 2),
                "count":     len(ids_in_zone)
            })

        # Annotate frame
        if detections.tracker_id is not None:
            labels = [f"ID:{t}" for t in detections.tracker_id]
        else:
            labels = []
        frame = box_ann.annotate(frame, detections)
        frame = label_ann.annotate(frame, detections, labels)

        for zone_name, z_cfg in zones.items():
            poly = z_cfg["polygon"]
            cv2.polylines(frame, [poly.astype(np.int32)], True, (0,255,0), 2)
            cx2, cy2 = poly.mean(axis=0).astype(int)
            cv2.putText(frame, zone_name, (max(cx2-60,0), max(cy2,20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1)

        out.write(frame)

        if frame_idx % 200 == 0:
            print(f".", end="")

    cap.release()
    out.release()

    # Build analytics summary
    dwell_summary = {}
    for zone, tracks in zone_dwell.items():
        dwell_secs = [f / fps for f in tracks.values()]
        dwell_summary[zone] = {
            "unique_visitors": len(tracks),
            "avg_dwell_sec":   round(float(np.mean(dwell_secs)), 2) if dwell_secs else 0,
            "max_dwell_sec":   round(float(max(dwell_secs)), 2)     if dwell_secs else 0,
        }

    peak_summary = {}
    for zone, frames in zone_counts.items():
        counts = [f["count"] for f in frames]
        peak_summary[zone] = {
            "peak_count": int(max(counts)) if counts else 0,
            "avg_count":  round(float(np.mean(counts)), 2) if counts else 0,
        }

    print(f" {len(events)} events")
    return events, {"dwell": dwell_summary, "peak": peak_summary}


# ── Run all cameras for both stores ────────────────────────────────
all_events  = []
all_analytics = {}
base_ts = datetime(2026, 4, 10, 18, 10, 0)

for store_key, store_cfg in STORE_CONFIG.items():
    store_id = store_cfg["store_id"]
    print(f"\n{'='*50}")
    print(f" Processing {store_key} ({store_id})")
    print(f"{'='*50}")
    all_analytics[store_id] = {}

    for cam_id, cam_cfg in store_cfg["cameras"].items():
        events, analytics = process_camera(store_id, cam_id, cam_cfg, base_ts)
        all_events.extend(events)
        all_analytics[store_id][cam_id] = analytics

print(f"\n\n DONE! Total events generated: {len(all_events)}")
event_types = {}
for e in all_events:
    t = e["event_type"]
    event_types[t] = event_types.get(t, 0) + 1
print("Event breakdown:", event_types)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

 Processing ST1 (ST1008)
   CAM1 (CAM 1 - zone.mp4)... 

...... 16 events
   CAM2 (CAM 2 - zone.mp4)... ...... 287 events
   CAM3 (CAM 3 - entry.mp4)... ....... 190 events
   CAM5 (CAM 5 - billing.mp4)... ..... 8 events

 Processing ST2 (ST1009)
   CAM1 (zone.mp4)... .... 163 events
   CAM_E1 (entry 1.mp4)... .... 411 events
   CAM_E2 (entry 2.mp4)... ... 686 events
   CAM_B (billing_area.mp4)... ..... 81 events


 DONE! Total events generated: 1842
Event breakdown: {'zone_entered': 437, 'zone_exited': 432, 'entry': 443, 'exit': 441, 'queue_completed': 89}


In [9]:
import json, pandas as pd

# Save events JSONL
with open("generated_events.jsonl", "w") as f:
    for event in all_events:
        f.write(json.dumps(event) + "\n")
print(f"generated_events.jsonl — {len(all_events)} events")

# Save analytics JSON
with open("store_analytics.json", "w") as f:
    json.dump(all_analytics, f, indent=2)
print("store_analytics.json")

# Sales analysis
df = pd.read_csv("POS - sample transactionsb1e826f.csv")
df['hour'] = df['order_time'].str[:2].astype(int)
hourly = df.groupby('hour').agg(
    orders=('order_id','nunique'),
    revenue=('total_amount','sum')
).reset_index()
brand_perf = df.groupby('brand_name')['total_amount'].sum().sort_values(ascending=False).reset_index()

sales_data = {
    "total_revenue": round(float(df['total_amount'].sum()), 2),
    "total_orders":  int(df['order_id'].nunique()),
    "hourly":        hourly.to_dict(orient='records'),
    "top_brands":    brand_perf.head(10).to_dict(orient='records'),
}

with open("sales_analytics.json", "w") as f:
    json.dump(sales_data, f, indent=2)
print(f"✅ sales_analytics.json — ₹{sales_data['total_revenue']:,.0f} total revenue")

# Anomaly detection
anomalies = []

for store_id, cams in all_analytics.items():
    for cam_id, analytics in cams.items():
        for zone, peak in analytics.get("peak", {}).items():
            if peak["peak_count"] >= 4:
                anomalies.append({
                    "type": "HIGH_CROWD_DENSITY",
                    "severity": "HIGH" if peak["peak_count"] >= 5 else "MEDIUM",
                    "store_id": store_id, "camera_id": cam_id, "zone": zone,
                    "peak_count": peak["peak_count"],
                    "message": f"Peak {peak['peak_count']} people in {zone}"
                })
        for zone, dwell in analytics.get("dwell", {}).items():
            if dwell["avg_dwell_sec"] > 45:
                anomalies.append({
                    "type": "HIGH_DWELL_TIME",
                    "severity": "INFO",
                    "store_id": store_id, "camera_id": cam_id, "zone": zone,
                    "avg_dwell_sec": dwell["avg_dwell_sec"],
                    "message": f"High avg dwell {dwell['avg_dwell_sec']}s in {zone}"
                })
            if dwell["unique_visitors"] == 0:
                anomalies.append({
                    "type": "DEAD_ZONE",
                    "severity": "LOW",
                    "store_id": store_id, "camera_id": cam_id, "zone": zone,
                    "message": f"Zero visitors in {zone}"
                })

# Queue abandoned anomalies
abandoned = [e for e in all_events if e.get("event_type") == "queue_abandoned"]
if abandoned:
    anomalies.append({
        "type": "QUEUE_ABANDONMENT",
        "severity": "HIGH",
        "count": len(abandoned),
        "message": f"{len(abandoned)} customers abandoned billing queue"
    })

with open("anomalies.json", "w") as f:
    json.dump(anomalies, f, indent=2)
print(f"✅ anomalies.json — {len(anomalies)} anomalies")

generated_events.jsonl — 1842 events
store_analytics.json
✅ sales_analytics.json — ₹34,332 total revenue
✅ anomalies.json — 3 anomalies


In [11]:
from google.colab import files
import os

for fname in ["generated_events.jsonl","store_analytics.json",
              "sales_analytics.json","anomalies.json","dashboard.html"]:
    if os.path.exists(fname):
        files.download(fname)
        print(f"⬇️  {fname}")

for f in os.listdir("."):
    if f.startswith("output_") and f.endswith(".mp4"):
        files.download(f)
        print(f"⬇️  {f}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  generated_events.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

⬇️  output_ST1009_CAM_B.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  output_ST1009_CAM_E2.mp4
